# FloodShift 09 — Gaussian-Latent Variational Grounded VLM

This notebook trains the proposed **variational multimodal language model** on the accepted full grounded corpus from FloodShift-08B.

The segmentation detector is **not retrained** and the geographic test remains **locked**.

## Source corpus

Expected accepted archive:

```text
FloodShift_08B_ARCHIVE.zip
SHA256:
421bb74db41f90012b7e5d0f1b342ecfc467b657b12177806973f3bf2e13da20
```

Expected corpus:

```text
314 train scenes
34 held-out validation scenes
3140 train instructions
340 validation instructions
348 evidence panels
0 scene overlap
```

## Architecture

A frozen CLIP ViT encodes each evidence panel:

\[
v=f_{\mathrm{CLIP}}(I)
\]

A trainable Gaussian visual adaptor predicts:

\[
\mu(v),\quad \log\sigma^2(v)
\]

and samples:

\[
z=\mu+\sigma\odot\epsilon,\qquad \epsilon\sim\mathcal N(0,I)
\]

The latent is converted into visual soft tokens and prepended to FLAN-T5's encoder embeddings.

The text input contains deterministic FloodShift evidence plus the instruction. The model therefore learns grounded scientific language rather than replacing the flood detector.

The fixed loss is:

\[
\mathcal L=
\mathcal L_{\text{text}}
+\beta\mathcal L_{\text{KL}}
+\lambda_u\mathcal L_{\text{variance-align}}
+\lambda_a\mathcal L_{\text{aux}}
\]

The visual-only auxiliary heads predict:

- temporary-flood fraction,
- calibrated predictive entropy,
- optical attribution share.

Final validation includes **full**, **zero-vision**, and **shuffled-vision** ablations. We do not claim meaningful visual grounding unless the full model beats the ablations.


In [1]:
# ============================================================
# CELL: P18-C00 — INSTALL
# Kaggle: Internet ON
# ============================================================
!pip -q install -U "transformers>=4.46,<5" "peft>=0.13,<1" accelerate sentencepiece safetensors

print("P18-C00 COMPLETE ✅")


P18-C00 COMPLETE ✅


In [2]:
# ============================================================
# CELL: P18-C01 — ENVIRONMENT
# ============================================================
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import sys, gc, json, math, time, random, re, hashlib, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoImageProcessor,
    CLIPVisionModel,
    AutoModelForSeq2SeqLM,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = torch.cuda.is_available()
torch.backends.cudnn.benchmark = True

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_09_Variational_VLM")
SOURCE_ROOT = OUTPUT_ROOT / "_source08b"
CACHE_ROOT = OUTPUT_ROOT / "cache"
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoint"
METRIC_ROOT = OUTPUT_ROOT / "metrics"

for p in [OUTPUT_ROOT, SOURCE_ROOT, CACHE_ROOT, CHECKPOINT_ROOT, METRIC_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

EXPECTED_08B_ARCHIVE_SHA256 = (
    "421bb74db41f90012b7e5d0f1b342ecfc467b657b12177806973f3bf2e13da20"
)

VISION_MODEL_NAME = "openai/clip-vit-base-patch32"
TEXT_MODEL_NAME = "google/flan-t5-base"

print("=" * 112)
print("P18-C01 — VARIATIONAL VLM ENVIRONMENT")
print("=" * 112)
print("Python       :", sys.version.split()[0])
print("NumPy        :", np.__version__)
print("Pandas       :", pd.__version__)
print("PyTorch      :", torch.__version__)
print("Device       :", DEVICE)
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
print("AMP          :", AMP_ENABLED)
print("Vision model :", VISION_MODEL_NAME)
print("Text model   :", TEXT_MODEL_NAME)
print("Output root  :", OUTPUT_ROOT)


P18-C01 — VARIATIONAL VLM ENVIRONMENT
Python       : 3.12.13
NumPy        : 2.0.2
Pandas       : 2.3.3
PyTorch      : 2.10.0+cu128
Device       : cuda:0
GPU          : Tesla T4
AMP          : True
Vision model : openai/clip-vit-base-patch32
Text model   : google/flan-t5-base
Output root  : /kaggle/working/FloodShift_09_Variational_VLM


In [3]:
# ============================================================
# CELL: P18-C02 — LOCATE / VERIFY FLOODSHIFT-08B CORPUS
# Supports actual ZIP or Kaggle auto-extracted contents.
# ============================================================
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def find_all(filename):
    return sorted(
        INPUT_ROOT.rglob(filename),
        key=lambda p: (len(str(p)), str(p)),
    )

def find_one(filename, required=True):
    matches = find_all(filename)

    if not matches:
        if required:
            raise FileNotFoundError(
                f"Could not find '{filename}' under /kaggle/input."
            )
        return None

    return matches[0]

ARCHIVE_08B = find_one("FloodShift_08B_ARCHIVE.zip", required=False)

if ARCHIVE_08B is not None:
    ARCHIVE_08B_SHA = sha256_file(ARCHIVE_08B)

    assert ARCHIVE_08B_SHA == EXPECTED_08B_ARCHIVE_SHA256, (
        "FloodShift-08B archive SHA mismatch.\n"
        f"Expected: {EXPECTED_08B_ARCHIVE_SHA256}\n"
        f"Found   : {ARCHIVE_08B_SHA}"
    )

    with zipfile.ZipFile(ARCHIVE_08B, "r") as zf:
        zf.extractall(SOURCE_ROOT)

    CORPUS_MODE = "verified_zip"
    CORPUS_BASE = SOURCE_ROOT

else:
    protocol_match = find_one("FloodShift_08B_protocol_record.json")

    CORPUS_BASE = protocol_match.parent
    ARCHIVE_08B_SHA = EXPECTED_08B_ARCHIVE_SHA256
    CORPUS_MODE = "kaggle_auto_extracted"

def locate_corpus_file(filename):
    matches = list(CORPUS_BASE.rglob(filename))

    if matches:
        return sorted(matches, key=lambda p: (len(str(p)), str(p)))[0]

    return find_one(filename)

TRAIN_JSONL = locate_corpus_file("FloodShift_08B_train.jsonl")
VAL_JSONL = locate_corpus_file("FloodShift_08B_validation.jsonl")
TRAIN_PACKETS_JSONL = locate_corpus_file("FloodShift_08B_train_scene_packets.jsonl")
VAL_PACKETS_JSONL = locate_corpus_file("FloodShift_08B_validation_scene_packets.jsonl")
DATASET_CARD_PATH = locate_corpus_file("FloodShift_08B_dataset_card.json")
PROTOCOL_PATH = locate_corpus_file("FloodShift_08B_protocol_record.json")
FILE_MANIFEST_PATH = locate_corpus_file("FloodShift_08B_file_manifest.csv")

with open(DATASET_CARD_PATH, "r") as f:
    dataset_card = json.load(f)

with open(PROTOCOL_PATH, "r") as f:
    protocol08b = json.load(f)

assert dataset_card["source_segmenter"] == "FloodShift-04H-CAL"
assert dataset_card["training_scenes"] == 314
assert dataset_card["validation_scenes"] == 34
assert dataset_card["training_examples"] == 3140
assert dataset_card["validation_examples"] == 340
assert dataset_card["scene_overlap_train_validation"] == 0
assert dataset_card["ground_truth_in_language_packets"] is False
assert dataset_card["frozen_geographic_test_scenes"] == 58
assert dataset_card["test_status"] == "LOCKED_NOT_EVALUATED"

assert protocol08b["05b_status"] == "NOT_PROMOTED_NOT_USED"
assert protocol08b["test_status"] == "LOCKED_NOT_EVALUATED"

# Strong verification for Kaggle auto-extracted form.
manifest08b = pd.read_csv(FILE_MANIFEST_PATH)
manifest_failures = []

for _, row in manifest08b.iterrows():
    rel = Path(str(row["relative_path"]))
    expected_sha = str(row["sha256"])

    candidate = CORPUS_BASE / rel

    if not candidate.exists():
        matches = list(INPUT_ROOT.rglob(rel.name))

        if len(matches) == 1:
            candidate = matches[0]

        elif len(matches) > 1:
            suffix_matches = [
                p for p in matches
                if str(p).endswith(str(rel))
            ]
            candidate = suffix_matches[0] if suffix_matches else matches[0]

        else:
            manifest_failures.append((str(rel), "missing"))
            continue

    actual_sha = sha256_file(candidate)

    if actual_sha != expected_sha:
        manifest_failures.append((str(rel), "sha_mismatch"))

assert not manifest_failures, (
    f"08B manifest verification failed: {manifest_failures[:10]}"
)

print("=" * 112)
print("P18-C02 — FLOODSHIFT-08B VERIFIED")
print("=" * 112)
print("Corpus mode          :", CORPUS_MODE)
print("Accepted archive SHA :", ARCHIVE_08B_SHA)
print("Manifest rows        :", len(manifest08b))
print("Manifest failures    :", len(manifest_failures))
print("Train scenes         :", dataset_card["training_scenes"])
print("Validation scenes    :", dataset_card["validation_scenes"])
print("Train examples       :", dataset_card["training_examples"])
print("Validation examples  :", dataset_card["validation_examples"])
print("Frozen geo test      :", dataset_card["test_status"])
print("P18-C02 COMPLETE ✅")


P18-C02 — FLOODSHIFT-08B VERIFIED
Corpus mode          : kaggle_auto_extracted
Accepted archive SHA : 421bb74db41f90012b7e5d0f1b342ecfc467b657b12177806973f3bf2e13da20
Manifest rows        : 359
Manifest failures    : 0
Train scenes         : 314
Validation scenes    : 34
Train examples       : 3140
Validation examples  : 340
Frozen geo test      : LOCKED_NOT_EVALUATED
P18-C02 COMPLETE ✅


In [4]:
# ============================================================
# CELL: P18-C03 — LOAD EXAMPLES + PACKETS + IMAGE INDEX
# ============================================================
def read_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                rows.append(json.loads(line))

    return rows

train_examples = read_jsonl(TRAIN_JSONL)
val_examples = read_jsonl(VAL_JSONL)
train_packets = read_jsonl(TRAIN_PACKETS_JSONL)
val_packets = read_jsonl(VAL_PACKETS_JSONL)

assert len(train_examples) == 3140
assert len(val_examples) == 340
assert len(train_packets) == 314
assert len(val_packets) == 34

packets = train_packets + val_packets

packet_by_scene = {
    p["scene_id"]: p
    for p in packets
}

assert len(packet_by_scene) == 348

train_scene_ids = {
    x["scene_id"]
    for x in train_examples
}

val_scene_ids = {
    x["scene_id"]
    for x in val_examples
}

assert len(train_scene_ids) == 314
assert len(val_scene_ids) == 34
assert train_scene_ids.isdisjoint(val_scene_ids)

all_images = list(CORPUS_BASE.rglob("*_evidence.png"))

if len(all_images) < 348:
    all_images = list(INPUT_ROOT.rglob("*_evidence.png"))

image_by_scene = {}

for p in all_images:
    name = p.name

    if not name.endswith("_evidence.png"):
        continue

    sid = name[:-len("_evidence.png")]

    if sid in packet_by_scene and sid not in image_by_scene:
        image_by_scene[sid] = p

assert len(image_by_scene) == 348, (
    f"Expected 348 evidence images, found {len(image_by_scene)}"
)

print("=" * 112)
print("P18-C03 — CORPUS LOADED")
print("=" * 112)
print("Training examples   :", len(train_examples))
print("Validation examples :", len(val_examples))
print("Unique images       :", len(image_by_scene))
print("Scene overlap       :", len(train_scene_ids & val_scene_ids))
print("P18-C03 COMPLETE ✅")


P18-C03 — CORPUS LOADED
Training examples   : 3140
Validation examples : 340
Unique images       : 348
Scene overlap       : 0
P18-C03 COMPLETE ✅


In [5]:
# ============================================================
# CELL: P18-C04 — STRUCTURED EVIDENCE TEXT
# ============================================================
def fmt_pct(x):
    if x is None:
        return "unavailable"
    return f"{100.0 * float(x):.2f}%"

def fmt_dec(x):
    if x is None:
        return "unavailable"
    return f"{float(x):.3f}"

def fmt_area(x):
    if x is None:
        return "unavailable"
    return f"{float(x):.2f} km²"

def build_evidence_text(packet):
    seg = packet["segmentation"]
    unc = packet["uncertainty"]
    sq = packet["sensor_quality"]
    att = packet["counterfactual_attribution"]
    geo = packet["spatial_geometry"]

    fields = [
        ("temporary_flood_fraction",
         fmt_pct(seg["temporary_flood"]["fraction_of_coverage"])),
        ("temporary_flood_area",
         fmt_area(seg["temporary_flood"]["area_km2"])),
        ("permanent_water_fraction",
         fmt_pct(seg["permanent_water"]["fraction_of_coverage"])),
        ("mean_predictive_entropy",
         fmt_dec(unc["mean_entropy"])),
        ("review10_fraction",
         fmt_pct(unc["review10_fraction"])),
        ("review20_fraction",
         fmt_pct(unc["review20_fraction"])),
        ("mean_pclear",
         fmt_dec(sq["mean_pclear"])),
        ("low_pclear_fraction",
         fmt_pct(sq["low_pclear_fraction"])),
        ("sar_attribution_share",
         fmt_pct(att["sar_share"])),
        ("optical_attribution_share",
         fmt_pct(att["optical_share"])),
        ("sensor_dominance",
         str(att["dominance"])),
        ("flip_rate_no_sar",
         fmt_pct(att["flip_rate_no_sar"])),
        ("flip_rate_no_optical",
         fmt_pct(att["flip_rate_no_optical"])),
        ("temporary_js_sar",
         fmt_dec(att["temporary_flood_js_sar"])),
        ("temporary_js_optical",
         fmt_dec(att["temporary_flood_js_optical"])),
        ("high_fragility_tile_fraction",
         fmt_pct(att["high_fragility_tile_fraction"])),
        ("temporary_components_8conn",
         str(int(geo["temporary_flood_connected_components_8conn"]))),
        ("largest_temp_component_area",
         fmt_area(geo["largest_temporary_component_area_km2"])),
    ]

    return " ; ".join(
        f"{k}={v}"
        for k, v in fields
    )

SYSTEM_PREFIX = (
    "You are FloodShift's grounded scientific assistant. "
    "Use only the verified evidence supplied below. "
    "Do not invent unsupported numerical values, physical causes, "
    "human impacts, or future forecasts. "
    "Counterfactual sensor attribution describes model dependence, "
    "not physical causality."
)

def build_source_text(example):
    packet = packet_by_scene[example["scene_id"]]
    evidence = build_evidence_text(packet)

    return (
        SYSTEM_PREFIX
        + "\nEVIDENCE: "
        + evidence
        + "\nQUESTION: "
        + example["instruction"]
    )

def scene_aux_targets(scene_id):
    packet = packet_by_scene[scene_id]

    return np.asarray(
        [
            float(
                packet["segmentation"]["temporary_flood"]["fraction_of_coverage"]
            ),
            float(
                packet["uncertainty"]["mean_entropy"]
            ),
            float(
                packet["counterfactual_attribution"]["optical_share"]
            ),
        ],
        dtype=np.float32,
    )

preview = train_examples[0]

print("=" * 112)
print("P18-C04 — GROUNDED INPUT PREVIEW")
print("=" * 112)
print(build_source_text(preview))
print()
print("TARGET:")
print(preview["response"])
print()
print(
    "Visual-only targets [temp_fraction, entropy, optical_share]:",
    scene_aux_targets(preview["scene_id"]),
)
print("P18-C04 COMPLETE ✅")


P18-C04 — GROUNDED INPUT PREVIEW
You are FloodShift's grounded scientific assistant. Use only the verified evidence supplied below. Do not invent unsupported numerical values, physical causes, human impacts, or future forecasts. Counterfactual sensor attribution describes model dependence, not physical causality.
EVIDENCE: temporary_flood_fraction=41.54% ; temporary_flood_area=4.07 km² ; permanent_water_fraction=1.46% ; mean_predictive_entropy=0.174 ; review10_fraction=33.49% ; review20_fraction=56.90% ; mean_pclear=0.674 ; low_pclear_fraction=30.73% ; sar_attribution_share=38.46% ; optical_attribution_share=61.54% ; sensor_dominance=Balanced ; flip_rate_no_sar=33.24% ; flip_rate_no_optical=42.94% ; temporary_js_sar=0.513 ; temporary_js_optical=0.855 ; high_fragility_tile_fraction=66.67% ; temporary_components_8conn=35 ; largest_temp_component_area=2.36 km²
QUESTION: Using only the verified FloodShift evidence, summarize the predicted temporary-flood and permanent-water extent in this 

In [6]:
# ============================================================
# CELL: P18-C05 — FROZEN CLIP FEATURE CACHE
# ============================================================
image_processor = AutoImageProcessor.from_pretrained(
    VISION_MODEL_NAME
)

vision_model = CLIPVisionModel.from_pretrained(
    VISION_MODEL_NAME
).to(DEVICE).eval()

for p in vision_model.parameters():
    p.requires_grad = False

@torch.inference_mode()
def encode_panel(path):
    with Image.open(path) as img:
        img = img.convert("RGB")

        batch = image_processor(
            images=img,
            return_tensors="pt",
        )

    pixel_values = batch["pixel_values"].to(DEVICE)

    with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
        out = vision_model(
            pixel_values=pixel_values
        )

    return out.pooler_output[0].float().cpu()

FEATURE_CACHE_PATH = CACHE_ROOT / "FloodShift_09_CLIP_features.pt"

vision_features = {}

t0 = time.time()

for idx, sid in enumerate(sorted(image_by_scene), start=1):
    vision_features[sid] = encode_panel(
        image_by_scene[sid]
    )

    if idx == 1 or idx % 50 == 0 or idx == len(image_by_scene):
        print(
            f"[{idx:03d}/{len(image_by_scene)}] CLIP features"
        )

feature_dim = int(
    next(iter(vision_features.values())).numel()
)

assert len(vision_features) == 348

torch.save(
    {
        "vision_model_name": VISION_MODEL_NAME,
        "feature_dim": feature_dim,
        "features": vision_features,
    },
    FEATURE_CACHE_PATH,
)

del vision_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 112)
print("P18-C05 — CLIP CACHE")
print("=" * 112)
print("Scenes      :", len(vision_features))
print("Feature dim :", feature_dim)
print("Saved       :", FEATURE_CACHE_PATH)
print("Minutes     :", round((time.time() - t0) / 60.0, 2))
print("P18-C05 COMPLETE ✅")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[001/348] CLIP features
[050/348] CLIP features
[100/348] CLIP features
[150/348] CLIP features
[200/348] CLIP features
[250/348] CLIP features
[300/348] CLIP features
[348/348] CLIP features
P18-C05 — CLIP CACHE
Scenes      : 348
Feature dim : 768
Saved       : /kaggle/working/FloodShift_09_Variational_VLM/cache/FloodShift_09_CLIP_features.pt
Minutes     : 0.45
P18-C05 COMPLETE ✅


In [7]:
# ============================================================
# CELL: P18-C06 — TOKENIZER / DATASET / COLLATOR
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL_NAME
)

MAX_SOURCE_LENGTH = 384
MAX_TARGET_LENGTH = 192

class GroundedVLMDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        sid = ex["scene_id"]

        return {
            "id": ex["id"],
            "scene_id": sid,
            "task": ex["task"],
            "response_mode": ex["response_mode"],
            "source_text": build_source_text(ex),
            "target_text": ex["response"],
            "vision_feature": vision_features[sid].clone(),
            "aux_target": torch.from_numpy(
                scene_aux_targets(sid)
            ),
        }

class GroundedVLMCollator:
    def __call__(self, rows):
        source_texts = [
            x["source_text"]
            for x in rows
        ]

        target_texts = [
            x["target_text"]
            for x in rows
        ]

        source = tokenizer(
            source_texts,
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LENGTH,
            return_tensors="pt",
        )

        target = tokenizer(
            text_target=target_texts,
            padding=True,
            truncation=True,
            max_length=MAX_TARGET_LENGTH,
            return_tensors="pt",
        )

        labels = target["input_ids"]
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            "ids": [x["id"] for x in rows],
            "scene_ids": [x["scene_id"] for x in rows],
            "tasks": [x["task"] for x in rows],
            "response_modes": [x["response_mode"] for x in rows],
            "source_texts": source_texts,
            "target_texts": target_texts,
            "input_ids": source["input_ids"],
            "attention_mask": source["attention_mask"],
            "labels": labels,
            "vision_features": torch.stack(
                [x["vision_feature"] for x in rows],
                dim=0,
            ),
            "aux_targets": torch.stack(
                [x["aux_target"] for x in rows],
                dim=0,
            ),
        }

train_dataset = GroundedVLMDataset(train_examples)
val_dataset = GroundedVLMDataset(val_examples)
collator = GroundedVLMCollator()

TRAIN_BATCH_SIZE = 2
VAL_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8
NUM_WORKERS = 0

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    num_workers=NUM_WORKERS,
    collate_fn=collator,
    generator=generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    collate_fn=collator,
)

batch = next(iter(train_loader))

print("=" * 112)
print("P18-C06 — DATA LOADERS")
print("=" * 112)
print("Train examples    :", len(train_dataset))
print("Validation        :", len(val_dataset))
print("Physical batch    :", TRAIN_BATCH_SIZE)
print("Grad accumulation :", GRAD_ACCUM_STEPS)
print("Effective batch   :", TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)
print("Vision feature    :", tuple(batch["vision_features"].shape))
print("Input IDs         :", tuple(batch["input_ids"].shape))
print("Labels            :", tuple(batch["labels"].shape))
print("P18-C06 COMPLETE ✅")


P18-C06 — DATA LOADERS
Train examples    : 3140
Validation        : 340
Physical batch    : 2
Grad accumulation : 8
Effective batch   : 16
Vision feature    : (2, 768)
Input IDs         : (2, 316)
Labels            : (2, 71)
P18-C06 COMPLETE ✅


In [8]:
# ============================================================
# CELL: P18-C07 — FLAN-T5 + LoRA
# ============================================================
text_model = AutoModelForSeq2SeqLM.from_pretrained(
    TEXT_MODEL_NAME
)

text_model.config.use_cache = False

if hasattr(text_model, "gradient_checkpointing_enable"):
    text_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q", "v"],
    bias="none",
)

text_model = get_peft_model(
    text_model,
    lora_config,
).to(DEVICE)

trainable_text = sum(
    p.numel()
    for p in text_model.parameters()
    if p.requires_grad
)

total_text = sum(
    p.numel()
    for p in text_model.parameters()
)

print("=" * 112)
print("P18-C07 — TEXT BACKBONE")
print("=" * 112)
print("Model             :", TEXT_MODEL_NAME)
print("Total parameters  :", f"{total_text:,}")
print("LoRA trainable    :", f"{trainable_text:,}")
print("Trainable %       :", f"{100 * trainable_text / total_text:.4f}%")
print("P18-C07 COMPLETE ✅")


P18-C07 — TEXT BACKBONE
Model             : google/flan-t5-base
Total parameters  : 248,462,592
LoRA trainable    : 884,736
Trainable %       : 0.3561%
P18-C07 COMPLETE ✅


In [9]:
# ============================================================
# PATCH — REMOVE INCOMPATIBLE KAGGLE TORCHAO
# ============================================================

!pip uninstall -y torchao

print("✅ Removed incompatible torchao")
print("⚠️ Now RESTART THE KAGGLE SESSION/KERNEL once.")

✅ Removed incompatible torchao
⚠️ Now RESTART THE KAGGLE SESSION/KERNEL once.


In [13]:
# ============================================================
# CELL: P18-C08 — GAUSSIAN VISUAL BOTTLENECK
# ============================================================
LATENT_DIM = 256
NUM_VISUAL_TOKENS = 4
BRIDGE_HIDDEN = 512

LOGVAR_MIN = -6.0
LOGVAR_MAX = 2.0

BETA_KL = 1e-4
LAMBDA_VAR_ALIGN = 0.10
LAMBDA_AUX = 0.20

class GaussianVisualBridge(nn.Module):
    def __init__(
        self,
        vision_dim,
        text_dim,
        latent_dim=LATENT_DIM,
        num_visual_tokens=NUM_VISUAL_TOKENS,
    ):
        super().__init__()

        self.latent_dim = int(latent_dim)
        self.num_visual_tokens = int(num_visual_tokens)

        self.shared = nn.Sequential(
            nn.LayerNorm(vision_dim),
            nn.Linear(vision_dim, BRIDGE_HIDDEN),
            nn.GELU(),
            nn.Dropout(0.10),
        )

        self.mu = nn.Linear(
            BRIDGE_HIDDEN,
            latent_dim,
        )

        self.logvar = nn.Linear(
            BRIDGE_HIDDEN,
            latent_dim,
        )

        self.to_visual_tokens = nn.Sequential(
            nn.Linear(
                latent_dim,
                num_visual_tokens * text_dim,
            ),
            nn.Tanh(),
        )

        self.token_norm = nn.LayerNorm(
            text_dim
        )

        self.aux_head = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.GELU(),
            nn.Linear(128, 3),
        )

    def encode(self, vision_features):
        h = self.shared(
            vision_features.float()
        )

        mu = self.mu(h)

        logvar = self.logvar(h).clamp(
            LOGVAR_MIN,
            LOGVAR_MAX,
        )

        return mu, logvar

    def reparameterize(
        self,
        mu,
        logvar,
        sample=True,
    ):
        if not sample:
            return mu

        std = torch.exp(
            0.5 * logvar
        )

        eps = torch.randn_like(std)

        return mu + std * eps

    def make_tokens(
        self,
        z,
        text_dim,
    ):
        tokens = self.to_visual_tokens(z)

        tokens = tokens.view(
            z.shape[0],
            self.num_visual_tokens,
            text_dim,
        )

        return self.token_norm(tokens)

    def forward(
        self,
        vision_features,
        text_dim,
        sample=True,
    ):
        mu, logvar = self.encode(
            vision_features
        )

        z = self.reparameterize(
            mu,
            logvar,
            sample=sample,
        )

        visual_tokens = self.make_tokens(
            z,
            text_dim,
        )

        aux_pred = torch.sigmoid(
            self.aux_head(mu)
        )

        variance = torch.exp(logvar)

        latent_uncertainty = (
            1.0
            - torch.exp(
                -variance.mean(dim=1)
            )
        )

        return {
            "mu": mu,
            "logvar": logvar,
            "z": z,
            "visual_tokens": visual_tokens,
            "aux_pred": aux_pred,
            "latent_uncertainty": latent_uncertainty,
        }

TEXT_DIM = int(
    text_model.config.d_model
)

visual_bridge = GaussianVisualBridge(
    vision_dim=feature_dim,
    text_dim=TEXT_DIM,
).to(DEVICE)

bridge_params = sum(
    p.numel()
    for p in visual_bridge.parameters()
    if p.requires_grad
)

print("=" * 112)
print("P18-C08 — VARIATIONAL BRIDGE")
print("=" * 112)
print("Vision dim       :", feature_dim)
print("Text dim         :", TEXT_DIM)
print("Latent dim       :", LATENT_DIM)
print("Visual tokens    :", NUM_VISUAL_TOKENS)
print("Trainable bridge :", f"{bridge_params:,}")
print("KL beta          :", BETA_KL)
print("Variance align λ :", LAMBDA_VAR_ALIGN)
print("Auxiliary λ      :", LAMBDA_AUX)
print("P18-C08 COMPLETE ✅")


P18-C08 — VARIATIONAL BRIDGE
Vision dim       : 768
Text dim         : 768
Latent dim       : 256
Visual tokens    : 4
Trainable bridge : 1,482,243
KL beta          : 0.0001
Variance align λ : 0.1
Auxiliary λ      : 0.2
P18-C08 COMPLETE ✅


In [14]:
# ============================================================
# CELL: P18-C09 — MULTIMODAL FORWARD / LOSS
# ============================================================
def prepare_multimodal_embeddings(
    input_ids,
    attention_mask,
    vision_features,
    sample_latent,
    vision_mode="full",
):
    text_embeddings = (
        text_model
        .get_input_embeddings()(
            input_ids
        )
    )

    bridge_out = visual_bridge(
        vision_features,
        text_dim=TEXT_DIM,
        sample=sample_latent,
    )

    visual_tokens = bridge_out["visual_tokens"]

    if vision_mode == "zero":
        visual_tokens = torch.zeros_like(
            visual_tokens
        )

    elif vision_mode == "shuffled":
        if visual_tokens.shape[0] > 1:
            visual_tokens = torch.roll(
                visual_tokens,
                shifts=1,
                dims=0,
            )

    elif vision_mode != "full":
        raise ValueError(
            vision_mode
        )

    inputs_embeds = torch.cat(
        [
            visual_tokens,
            text_embeddings,
        ],
        dim=1,
    )

    visual_mask = torch.ones(
        (
            input_ids.shape[0],
            NUM_VISUAL_TOKENS,
        ),
        dtype=attention_mask.dtype,
        device=attention_mask.device,
    )

    multimodal_mask = torch.cat(
        [
            visual_mask,
            attention_mask,
        ],
        dim=1,
    )

    return (
        inputs_embeds,
        multimodal_mask,
        bridge_out,
    )

def compute_losses(
    batch,
    sample_latent,
    vision_mode="full",
):
    input_ids = batch["input_ids"].to(
        DEVICE,
        non_blocking=True,
    )

    attention_mask = batch["attention_mask"].to(
        DEVICE,
        non_blocking=True,
    )

    labels = batch["labels"].to(
        DEVICE,
        non_blocking=True,
    )

    vision_feature = batch["vision_features"].to(
        DEVICE,
        non_blocking=True,
    )

    aux_target = batch["aux_targets"].to(
        DEVICE,
        non_blocking=True,
    ).float()

    (
        inputs_embeds,
        multimodal_mask,
        bridge_out,
    ) = prepare_multimodal_embeddings(
        input_ids,
        attention_mask,
        vision_feature,
        sample_latent=sample_latent,
        vision_mode=vision_mode,
    )

    outputs = text_model(
        inputs_embeds=inputs_embeds,
        attention_mask=multimodal_mask,
        labels=labels,
        return_dict=True,
    )

    text_loss = outputs.loss

    mu = bridge_out["mu"]
    logvar = bridge_out["logvar"]

    kl_loss = (
        -0.5
        * (
            1.0
            + logvar
            - mu.pow(2)
            - logvar.exp()
        )
    ).mean()

    entropy_target = aux_target[:, 1]

    variance_align_loss = F.mse_loss(
        bridge_out["latent_uncertainty"],
        entropy_target,
    )

    aux_loss = F.mse_loss(
        bridge_out["aux_pred"],
        aux_target,
    )

    total_loss = (
        text_loss
        + BETA_KL * kl_loss
        + LAMBDA_VAR_ALIGN * variance_align_loss
        + LAMBDA_AUX * aux_loss
    )

    return {
        "loss": total_loss,
        "text_loss": text_loss,
        "kl_loss": kl_loss,
        "variance_align_loss": variance_align_loss,
        "aux_loss": aux_loss,
        "aux_pred": bridge_out["aux_pred"],
        "aux_target": aux_target,
        "latent_uncertainty": bridge_out["latent_uncertainty"],
        "mu": mu,
        "logvar": logvar,
    }

smoke = next(iter(train_loader))

text_model.train()
visual_bridge.train()

with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
    smoke_out = compute_losses(
        smoke,
        sample_latent=True,
        vision_mode="full",
    )

for k in [
    "loss",
    "text_loss",
    "kl_loss",
    "variance_align_loss",
    "aux_loss",
]:
    assert torch.isfinite(smoke_out[k]).all(), (
        f"Non-finite smoke loss: {k}"
    )

print("=" * 112)
print("P18-C09 — LOSS SMOKE TEST")
print("=" * 112)
print("Total loss :", float(smoke_out["loss"].detach().cpu()))
print("Text loss  :", float(smoke_out["text_loss"].detach().cpu()))
print("KL         :", float(smoke_out["kl_loss"].detach().cpu()))
print("Var align  :", float(smoke_out["variance_align_loss"].detach().cpu()))
print("Aux        :", float(smoke_out["aux_loss"].detach().cpu()))
print("P18-C09 COMPLETE ✅")


AssertionError: Non-finite smoke loss: loss

In [15]:
# ============================================================
# PATCH — P18-C09 NUMERICAL STABILITY
# Diagnose inputs + disable FP16 for FLAN-T5 training
# ============================================================

# T4 + FLAN-T5 can occasionally produce non-finite activations
# under fp16 autocast. We use full FP32 for the trainable VLM.
AMP_ENABLED = False

print("=" * 110)
print("P18-C09 STABILITY PATCH")
print("=" * 110)
print("AMP_ENABLED =", AMP_ENABLED)
print()


# ------------------------------------------------------------
# 1. Fresh batch
# ------------------------------------------------------------
smoke = next(iter(train_loader))

vf = smoke["vision_features"]
aux = smoke["aux_targets"]
labels = smoke["labels"]

print("INPUT CHECKS")
print("-" * 60)

print(
    "Vision features finite :",
    bool(torch.isfinite(vf).all())
)

print(
    "Vision feature range   :",
    float(vf.min()),
    "to",
    float(vf.max())
)

print(
    "Aux targets finite     :",
    bool(torch.isfinite(aux).all())
)

print(
    "Aux targets            :",
    aux
)

valid_tokens = (
    labels != -100
).sum(dim=1)

print(
    "Valid target tokens    :",
    valid_tokens.tolist()
)

assert torch.isfinite(vf).all(), \
    "❌ CLIP feature cache contains NaN/Inf"

assert torch.isfinite(aux).all(), \
    "❌ Auxiliary target contains NaN/Inf"

assert (valid_tokens > 0).all(), \
    "❌ At least one training target contains only ignored tokens"


# ------------------------------------------------------------
# 2. Inspect Gaussian bridge BEFORE T5
# ------------------------------------------------------------
visual_bridge.eval()

with torch.no_grad():

    bridge_test = visual_bridge(
        vf.to(DEVICE),
        text_dim=TEXT_DIM,
        sample=False,
    )

print()
print("GAUSSIAN BRIDGE CHECKS")
print("-" * 60)

for name in [
    "mu",
    "logvar",
    "z",
    "visual_tokens",
    "aux_pred",
    "latent_uncertainty",
]:
    tensor = bridge_test[name]

    print(
        f"{name:<22}",
        "finite =",
        bool(torch.isfinite(tensor).all()),
        "| min =",
        float(tensor.min()),
        "| max =",
        float(tensor.max()),
    )

    assert torch.isfinite(tensor).all(), \
        f"❌ Non-finite values in {name}"


# ------------------------------------------------------------
# 3. Deterministic FP32 forward
# ------------------------------------------------------------
text_model.eval()
visual_bridge.eval()

with torch.no_grad():

    out_det = compute_losses(
        smoke,
        sample_latent=False,
        vision_mode="full",
    )

print()
print("DETERMINISTIC FP32 LOSS")
print("-" * 60)

for k in [
    "loss",
    "text_loss",
    "kl_loss",
    "variance_align_loss",
    "aux_loss",
]:

    value = float(
        out_det[k]
        .detach()
        .cpu()
    )

    print(
        f"{k:<25}: {value}"
    )

    assert math.isfinite(value), \
        f"❌ Still non-finite in FP32: {k}"


# ------------------------------------------------------------
# 4. Stochastic reparameterized FP32 forward
# ------------------------------------------------------------
with torch.no_grad():

    out_stochastic = compute_losses(
        smoke,
        sample_latent=True,
        vision_mode="full",
    )

print()
print("STOCHASTIC FP32 LOSS")
print("-" * 60)

for k in [
    "loss",
    "text_loss",
    "kl_loss",
    "variance_align_loss",
    "aux_loss",
]:

    value = float(
        out_stochastic[k]
        .detach()
        .cpu()
    )

    print(
        f"{k:<25}: {value}"
    )

    assert math.isfinite(value), \
        f"❌ Stochastic FP32 still non-finite: {k}"


# Restore train state
text_model.train()
visual_bridge.train()

print()
print("=" * 110)
print("✅ FP32 FORWARD IS FINITE")
print("✅ Keep AMP_ENABLED = False for Notebook 09")
print("✅ Continue with P18-C10")
print("=" * 110)

P18-C09 STABILITY PATCH
AMP_ENABLED = False

INPUT CHECKS
------------------------------------------------------------
Vision features finite : True
Vision feature range   : -7.224842071533203 to 3.7886760234832764
Aux targets finite     : True
Aux targets            : tensor([[0.0233, 0.0393, 0.9759],
        [0.0068, 0.0429, 0.7380]])
Valid target tokens    : [53, 71]

GAUSSIAN BRIDGE CHECKS
------------------------------------------------------------
mu                     finite = True | min = -0.6006504893302917 | max = 0.7285009026527405
logvar                 finite = True | min = -0.518714189529419 | max = 0.5731983184814453
z                      finite = True | min = -0.6006504893302917 | max = 0.7285009026527405
visual_tokens          finite = True | min = -3.4739766120910645 | max = 3.51568603515625
aux_pred               finite = True | min = 0.4702104926109314 | max = 0.516341507434845
latent_uncertainty     finite = True | min = 0.6370880603790283 | max = 0.6409784555435

In [16]:
# ============================================================
# CELL: P18-C10 — OPTIMIZER + FIXED TRAINING PROTOCOL
# ============================================================
EPOCHS = 3

LR_LORA = 1e-4
LR_BRIDGE = 3e-4

WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0
WARMUP_FRACTION = 0.10

optimizer = torch.optim.AdamW(
    [
        {
            "params": [
                p
                for p in text_model.parameters()
                if p.requires_grad
            ],
            "lr": LR_LORA,
            "name": "lora",
        },
        {
            "params": visual_bridge.parameters(),
            "lr": LR_BRIDGE,
            "name": "visual_bridge",
        },
    ],
    weight_decay=WEIGHT_DECAY,
)

optimizer_steps_per_epoch = math.ceil(
    len(train_loader)
    / GRAD_ACCUM_STEPS
)

total_optimizer_steps = (
    optimizer_steps_per_epoch
    * EPOCHS
)

warmup_steps = max(
    1,
    int(
        total_optimizer_steps
        * WARMUP_FRACTION
    ),
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimizer_steps,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED,
)

trainable_params = (
    sum(
        p.numel()
        for p in text_model.parameters()
        if p.requires_grad
    )
    + sum(
        p.numel()
        for p in visual_bridge.parameters()
        if p.requires_grad
    )
)

print("=" * 112)
print("P18-C10 — FIXED TRAINING PROTOCOL")
print("=" * 112)
print("Epochs                :", EPOCHS)
print("Train batches/epoch   :", len(train_loader))
print("Optimizer steps/epoch :", optimizer_steps_per_epoch)
print("Total optimizer steps :", total_optimizer_steps)
print("Warmup steps          :", warmup_steps)
print("LoRA LR               :", LR_LORA)
print("Bridge LR             :", LR_BRIDGE)
print("Trainable params      :", f"{trainable_params:,}")
print("P18-C10 COMPLETE ✅")


P18-C10 — FIXED TRAINING PROTOCOL
Epochs                : 3
Train batches/epoch   : 1570
Optimizer steps/epoch : 197
Total optimizer steps : 591
Warmup steps          : 59
LoRA LR               : 0.0001
Bridge LR             : 0.0003
Trainable params      : 2,366,979
P18-C10 COMPLETE ✅


In [17]:
# ============================================================
# CELL: P18-C11 — VALIDATION FUNCTION
# ============================================================
@torch.inference_mode()
def evaluate_loader(
    loader,
    vision_mode="full",
):
    text_model.eval()
    visual_bridge.eval()

    sums = {
        "loss": 0.0,
        "text_loss": 0.0,
        "kl_loss": 0.0,
        "variance_align_loss": 0.0,
        "aux_loss": 0.0,
    }

    n_batches = 0

    aux_pred_all = []
    aux_target_all = []
    latent_unc_all = []

    for batch in loader:
        with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
            out = compute_losses(
                batch,
                sample_latent=False,
                vision_mode=vision_mode,
            )

        for k in sums:
            sums[k] += float(
                out[k].detach().cpu()
            )

        aux_pred_all.append(
            out["aux_pred"].detach().cpu()
        )

        aux_target_all.append(
            out["aux_target"].detach().cpu()
        )

        latent_unc_all.append(
            out["latent_uncertainty"].detach().cpu()
        )

        n_batches += 1

    means = {
        k: v / max(n_batches, 1)
        for k, v in sums.items()
    }

    aux_pred = torch.cat(
        aux_pred_all,
        dim=0,
    ).numpy()

    aux_target = torch.cat(
        aux_target_all,
        dim=0,
    ).numpy()

    latent_unc = torch.cat(
        latent_unc_all,
        dim=0,
    ).numpy()

    means.update({
        "temp_fraction_MAE": float(
            np.mean(
                np.abs(
                    aux_pred[:, 0]
                    - aux_target[:, 0]
                )
            )
        ),
        "entropy_MAE": float(
            np.mean(
                np.abs(
                    aux_pred[:, 1]
                    - aux_target[:, 1]
                )
            )
        ),
        "optical_share_MAE": float(
            np.mean(
                np.abs(
                    aux_pred[:, 2]
                    - aux_target[:, 2]
                )
            )
        ),
        "latent_uncertainty_MAE": float(
            np.mean(
                np.abs(
                    latent_unc
                    - aux_target[:, 1]
                )
            )
        ),
    })

    if (
        np.std(latent_unc) > 1e-8
        and np.std(aux_target[:, 1]) > 1e-8
    ):
        means["latent_entropy_correlation"] = float(
            np.corrcoef(
                latent_unc,
                aux_target[:, 1],
            )[0, 1]
        )
    else:
        means["latent_entropy_correlation"] = float("nan")

    return means

pretrain_val = evaluate_loader(
    val_loader,
    vision_mode="full",
)

print("=" * 112)
print("P18-C11 — PRETRAIN VALIDATION")
print("=" * 112)

for k, v in pretrain_val.items():
    print(f"{k:<30}: {v:.6f}")

print("P18-C11 COMPLETE ✅")


P18-C11 — PRETRAIN VALIDATION
loss                          : 3.463686
text_loss                     : 3.405176
kl_loss                       : 0.033908
variance_align_loss           : 0.305931
aux_loss                      : 0.139568
temp_fraction_MAE             : 0.406772
entropy_MAE                   : 0.393705
optical_share_MAE             : 0.281551
latent_uncertainty_MAE        : 0.550874
latent_entropy_correlation    : 0.059439
P18-C11 COMPLETE ✅


In [18]:
# ============================================================
# CELL: P18-C12 — FIXED 3-EPOCH TRAINING
# ============================================================
BEST_PATH = (
    CHECKPOINT_ROOT
    / "FloodShift_09_best_adapter.pt"
)

history = []
best_val_loss = float("inf")
best_epoch = None
optimizer_steps = 0

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    text_model.train()
    visual_bridge.train()

    optimizer.zero_grad(
        set_to_none=True
    )

    running = {
        "loss": 0.0,
        "text_loss": 0.0,
        "kl_loss": 0.0,
        "variance_align_loss": 0.0,
        "aux_loss": 0.0,
    }

    counted = 0

    for bi, batch in enumerate(train_loader, start=1):
        with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
            out = compute_losses(
                batch,
                sample_latent=True,
                vision_mode="full",
            )

            loss = (
                out["loss"]
                / GRAD_ACCUM_STEPS
            )

        if not torch.isfinite(loss).all():
            raise FloatingPointError(
                f"Non-finite loss at epoch {epoch}, batch {bi}"
            )

        scaler.scale(loss).backward()

        for k in running:
            running[k] += float(
                out[k].detach().cpu()
            )

        counted += 1

        should_step = (
            bi % GRAD_ACCUM_STEPS == 0
            or bi == len(train_loader)
        )

        if should_step:
            scaler.unscale_(
                optimizer
            )

            grad_norm = (
                torch.nn.utils
                .clip_grad_norm_(
                    [
                        p
                        for p in list(text_model.parameters())
                        + list(visual_bridge.parameters())
                        if p.requires_grad
                    ],
                    MAX_GRAD_NORM,
                )
            )

            if not torch.isfinite(grad_norm):
                raise FloatingPointError(
                    f"Non-finite gradient norm at epoch {epoch}, batch {bi}"
                )

            scaler.step(
                optimizer
            )

            scaler.update()

            optimizer.zero_grad(
                set_to_none=True
            )

            scheduler.step()
            optimizer_steps += 1

        if (
            bi == 1
            or bi % 200 == 0
            or bi == len(train_loader)
        ):
            print(
                f"Epoch {epoch}/{EPOCHS} "
                f"[{bi:04d}/{len(train_loader)}] "
                f"loss={float(out['loss'].detach().cpu()):.4f} "
                f"text={float(out['text_loss'].detach().cpu()):.4f} "
                f"kl={float(out['kl_loss'].detach().cpu()):.4f}"
            )

    train_means = {
        k: v / max(counted, 1)
        for k, v in running.items()
    }

    val_metrics = evaluate_loader(
        val_loader,
        vision_mode="full",
    )

    row = {
        "epoch": epoch,
        "train": train_means,
        "validation": val_metrics,
        "optimizer_steps": optimizer_steps,
        "minutes": (
            time.time()
            - epoch_start
        ) / 60.0,
    }

    history.append(row)

    print("=" * 112)
    print(f"EPOCH {epoch}/{EPOCHS} COMPLETE")
    print("Train:", {
        k: round(v, 5)
        for k, v in train_means.items()
    })
    print("Validation:", {
        k: round(v, 5)
        for k, v in val_metrics.items()
    })
    print("Minutes:", round(row["minutes"], 2))
    print("=" * 112)

    if val_metrics["loss"] < best_val_loss:
        best_val_loss = float(
            val_metrics["loss"]
        )

        best_epoch = int(epoch)

        checkpoint = {
            "epoch": best_epoch,
            "validation_loss": best_val_loss,
            "visual_bridge_state_dict": {
                k: v.detach().cpu()
                for k, v in visual_bridge.state_dict().items()
            },
            "peft_state_dict": {
                k: v.detach().cpu()
                for k, v in get_peft_model_state_dict(
                    text_model
                ).items()
            },
            "config": {
                "vision_model_name": VISION_MODEL_NAME,
                "text_model_name": TEXT_MODEL_NAME,
                "feature_dim": feature_dim,
                "latent_dim": LATENT_DIM,
                "num_visual_tokens": NUM_VISUAL_TOKENS,
                "beta_kl": BETA_KL,
                "lambda_var_align": LAMBDA_VAR_ALIGN,
                "lambda_aux": LAMBDA_AUX,
                "lora_r": 8,
                "lora_alpha": 16,
            },
        }

        torch.save(
            checkpoint,
            BEST_PATH,
        )

        print(
            "✅ New best checkpoint:",
            BEST_PATH,
        )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print()
print("Training complete.")
print("Best epoch      :", best_epoch)
print("Best val loss   :", best_val_loss)
print("Optimizer steps :", optimizer_steps)
print("P18-C12 COMPLETE ✅")


Epoch 1/3 [0001/1570] loss=3.9938 text=3.9391 kl=0.0399
Epoch 1/3 [0200/1570] loss=3.9224 text=3.8601 kl=0.0435
Epoch 1/3 [0400/1570] loss=3.2987 text=3.2655 kl=0.1600
Epoch 1/3 [0600/1570] loss=3.1037 text=3.0864 kl=0.7029
Epoch 1/3 [0800/1570] loss=2.3805 text=2.3741 kl=1.4439
Epoch 1/3 [1000/1570] loss=2.5062 text=2.4924 kl=1.7909
Epoch 1/3 [1200/1570] loss=2.6076 text=2.6048 kl=2.0811
Epoch 1/3 [1400/1570] loss=2.4954 text=2.4923 kl=2.2219
Epoch 1/3 [1570/1570] loss=1.8112 text=1.8088 kl=2.6108
EPOCH 1/3 COMPLETE
Train: {'loss': 3.07615, 'text_loss': 3.05553, 'kl_loss': 1.19537, 'variance_align_loss': 0.12291, 'aux_loss': 0.04105}
Validation: {'loss': 1.58959, 'text_loss': 1.58771, 'kl_loss': 2.36514, 'variance_align_loss': 0.00292, 'aux_loss': 0.00676, 'temp_fraction_MAE': 0.06392, 'entropy_MAE': 0.04011, 'optical_share_MAE': 0.08558, 'latent_uncertainty_MAE': 0.04632, 'latent_entropy_correlation': -0.18448}
Minutes: 8.06
✅ New best checkpoint: /kaggle/working/FloodShift_09_Variat

In [19]:
# ============================================================
# CELL: P18-C13 — RELOAD BEST CHECKPOINT
# ============================================================
best_ckpt = torch.load(
    BEST_PATH,
    map_location="cpu",
    weights_only=False,
)

visual_bridge.load_state_dict(
    best_ckpt["visual_bridge_state_dict"],
    strict=True,
)

set_peft_model_state_dict(
    text_model,
    best_ckpt["peft_state_dict"],
)

text_model = text_model.to(
    DEVICE
).eval()

visual_bridge = visual_bridge.to(
    DEVICE
).eval()

print("=" * 112)
print("P18-C13 — BEST CHECKPOINT RELOADED")
print("=" * 112)
print("Epoch      :", best_ckpt["epoch"])
print("Val loss   :", best_ckpt["validation_loss"])
print("Checkpoint :", BEST_PATH)
print("P18-C13 COMPLETE ✅")


P18-C13 — BEST CHECKPOINT RELOADED
Epoch      : 3
Val loss   : 0.7407236281563254
Checkpoint : /kaggle/working/FloodShift_09_Variational_VLM/checkpoint/FloodShift_09_best_adapter.pt
P18-C13 COMPLETE ✅


In [20]:
# ============================================================
# CELL: P18-C14 — VISUAL-CONDITIONING ABLATION
# ============================================================
full_val = evaluate_loader(
    val_loader,
    vision_mode="full",
)

zero_val = evaluate_loader(
    val_loader,
    vision_mode="zero",
)

shuffled_val = evaluate_loader(
    val_loader,
    vision_mode="shuffled",
)

ablation_df = pd.DataFrame([
    {
        "vision_mode": "full",
        **full_val,
    },
    {
        "vision_mode": "zero",
        **zero_val,
    },
    {
        "vision_mode": "shuffled",
        **shuffled_val,
    },
])

print("=" * 112)
print("P18-C14 — VISUAL ABLATION")
print("=" * 112)

display(
    ablation_df[
        [
            "vision_mode",
            "loss",
            "text_loss",
            "temp_fraction_MAE",
            "entropy_MAE",
            "optical_share_MAE",
            "latent_uncertainty_MAE",
            "latent_entropy_correlation",
        ]
    ]
)

print()
print(
    "Full vs zero text-loss gain:",
    zero_val["text_loss"]
    - full_val["text_loss"],
)

print(
    "Full vs shuffled text-loss gain:",
    shuffled_val["text_loss"]
    - full_val["text_loss"],
)

print("P18-C14 COMPLETE ✅")


P18-C14 — VISUAL ABLATION


,vision_mode,loss,text_loss,temp_fraction_MAE,entropy_MAE,optical_share_MAE,latent_uncertainty_MAE,latent_entropy_correlation
0,full,0.740724,0.738877,0.068123,0.034593,0.088533,0.042667,0.038127
1,zero,0.750261,0.748415,0.068123,0.034593,0.088533,0.042667,0.038127
2,shuffled,0.740718,0.738872,0.068123,0.034593,0.088533,0.042667,0.038127



Full vs zero text-loss gain: 0.009537395308999441
Full vs shuffled text-loss gain: -5.827230565746966e-06
P18-C14 COMPLETE ✅


In [21]:
# ============================================================
# CELL: P18-C15 — DETERMINISTIC GENERATION
# ============================================================
@torch.inference_mode()
def generate_batch(
    batch,
    max_new_tokens=192,
):
    text_model.eval()
    visual_bridge.eval()

    input_ids = batch["input_ids"].to(
        DEVICE
    )

    attention_mask = batch["attention_mask"].to(
        DEVICE
    )

    vision_feature = batch["vision_features"].to(
        DEVICE
    )

    (
        inputs_embeds,
        multimodal_mask,
        bridge_out,
    ) = prepare_multimodal_embeddings(
        input_ids,
        attention_mask,
        vision_feature,
        sample_latent=False,
        vision_mode="full",
    )

    generated = text_model.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=multimodal_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
    )

    texts = tokenizer.batch_decode(
        generated,
        skip_special_tokens=True,
    )

    return texts, bridge_out

generation_rows = []

for bi, batch in enumerate(val_loader, start=1):
    preds, bridge_out = generate_batch(
        batch
    )

    for i, pred in enumerate(preds):
        generation_rows.append({
            "id": batch["ids"][i],
            "scene_id": batch["scene_ids"][i],
            "task": batch["tasks"][i],
            "response_mode": batch["response_modes"][i],
            "reference": batch["target_texts"][i],
            "prediction": pred,
            "source_text": batch["source_texts"][i],
        })

    if bi == 1 or bi % 20 == 0 or bi == len(val_loader):
        print(
            f"[{bi:03d}/{len(val_loader)}] generation"
        )

generation_df = pd.DataFrame(
    generation_rows
)

assert len(generation_df) == 340

print("=" * 112)
print("P18-C15 — GENERATION COMPLETE")
print("=" * 112)
print("Validation generations:", len(generation_df))
print("P18-C15 COMPLETE ✅")


[001/85] generation
[020/85] generation
[040/85] generation
[060/85] generation
[080/85] generation
[085/85] generation
P18-C15 — GENERATION COMPLETE
Validation generations: 340
P18-C15 COMPLETE ✅


In [22]:
# ============================================================
# CELL: P18-C16 — GROUNDED GENERATION METRICS
# ============================================================
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def token_f1(reference, prediction):
    ref = normalize_text(reference).split()
    pred = normalize_text(prediction).split()

    if not ref and not pred:
        return 1.0

    if not ref or not pred:
        return 0.0

    ref_counts = Counter(ref)
    pred_counts = Counter(pred)

    overlap = sum(
        (
            ref_counts
            & pred_counts
        ).values()
    )

    precision = overlap / max(len(pred), 1)
    recall = overlap / max(len(ref), 1)

    if precision + recall == 0:
        return 0.0

    return (
        2
        * precision
        * recall
        / (
            precision
            + recall
        )
    )

def lcs_length(a, b):
    if len(a) < len(b):
        a, b = b, a

    prev = [0] * (len(b) + 1)

    for x in a:
        cur = [0]

        for j, y in enumerate(b, start=1):
            if x == y:
                cur.append(
                    prev[j - 1] + 1
                )
            else:
                cur.append(
                    max(
                        cur[-1],
                        prev[j],
                    )
                )

        prev = cur

    return prev[-1]

def rouge_l_f1(reference, prediction):
    ref = normalize_text(reference).split()
    pred = normalize_text(prediction).split()

    if not ref and not pred:
        return 1.0

    if not ref or not pred:
        return 0.0

    lcs = lcs_length(ref, pred)

    precision = lcs / len(pred)
    recall = lcs / len(ref)

    if precision + recall == 0:
        return 0.0

    return (
        2
        * precision
        * recall
        / (
            precision
            + recall
        )
    )

NUMBER_RE = re.compile(
    r"[-+]?\d+(?:\.\d+)?"
)

def extract_numbers(text):
    return [
        float(x)
        for x in NUMBER_RE.findall(
            text
        )
    ]

def number_match(x, candidates):
    return any(
        math.isclose(
            x,
            y,
            rel_tol=0.0,
            abs_tol=0.011,
        )
        for y in candidates
    )

def numeric_recall(reference, prediction):
    ref_nums = extract_numbers(reference)
    pred_nums = extract_numbers(prediction)

    if not ref_nums:
        return 1.0

    matched = sum(
        number_match(
            x,
            pred_nums,
        )
        for x in ref_nums
    )

    return (
        matched
        / len(ref_nums)
    )

def unsupported_numeric_rate(
    source_text,
    prediction,
):
    allowed = extract_numbers(
        source_text
    )

    pred_nums = extract_numbers(
        prediction
    )

    if not pred_nums:
        return 0.0

    unsupported = sum(
        not number_match(
            x,
            allowed,
        )
        for x in pred_nums
    )

    return (
        unsupported
        / len(pred_nums)
    )

REFUSAL_MARKERS = [
    "does not",
    "cannot",
    "not contain",
    "not establish",
    "not provide",
    "cannot be determined",
    "would require",
]

def looks_like_refusal(text):
    t = normalize_text(text)

    return any(
        marker in t
        for marker in REFUSAL_MARKERS
    )

metric_rows = []

for _, row in generation_df.iterrows():
    ref = row["reference"]
    pred = row["prediction"]
    expected_mode = row["response_mode"]

    metric_rows.append({
        "id": row["id"],
        "scene_id": row["scene_id"],
        "task": row["task"],
        "response_mode": expected_mode,
        "exact_match": (
            normalize_text(ref)
            == normalize_text(pred)
        ),
        "token_f1": token_f1(
            ref,
            pred,
        ),
        "rouge_l_f1": rouge_l_f1(
            ref,
            pred,
        ),
        "numeric_recall": numeric_recall(
            ref,
            pred,
        ),
        "unsupported_numeric_rate": unsupported_numeric_rate(
            row["source_text"],
            pred,
        ),
        "looks_like_refusal": looks_like_refusal(
            pred
        ),
    })

generation_metrics_df = pd.DataFrame(
    metric_rows
)

refusal_rows = generation_metrics_df[
    generation_metrics_df["response_mode"]
    == "refuse"
]

answer_rows = generation_metrics_df[
    generation_metrics_df["response_mode"]
    == "answer"
]

summary_metrics = {
    "exact_match": float(
        generation_metrics_df["exact_match"].mean()
    ),
    "token_f1": float(
        generation_metrics_df["token_f1"].mean()
    ),
    "rouge_l_f1": float(
        generation_metrics_df["rouge_l_f1"].mean()
    ),
    "numeric_recall": float(
        generation_metrics_df["numeric_recall"].mean()
    ),
    "unsupported_numeric_rate": float(
        generation_metrics_df["unsupported_numeric_rate"].mean()
    ),
    "refusal_recall": float(
        refusal_rows["looks_like_refusal"].mean()
    ),
    "false_refusal_rate_on_answers": float(
        answer_rows["looks_like_refusal"].mean()
    ),
}

print("=" * 112)
print("P18-C16 — GROUNDED GENERATION METRICS")
print("=" * 112)

for k, v in summary_metrics.items():
    print(f"{k:<32}: {v:.6f}")

task_metric_df = (
    generation_metrics_df
    .groupby(
        "task",
        as_index=False,
    )
    .agg(
        examples=("id", "count"),
        token_f1=("token_f1", "mean"),
        rouge_l_f1=("rouge_l_f1", "mean"),
        numeric_recall=("numeric_recall", "mean"),
        unsupported_numeric_rate=("unsupported_numeric_rate", "mean"),
    )
)

print()
print("Task-level metrics:")
display(task_metric_df)

print("P18-C16 COMPLETE ✅")


P18-C16 — GROUNDED GENERATION METRICS
exact_match                     : 0.000000
token_f1                        : 0.448366
rouge_l_f1                      : 0.396140
numeric_recall                  : 0.681250
unsupported_numeric_rate        : 0.115686
refusal_recall                  : 0.970588
false_refusal_rate_on_answers   : 0.103139

Task-level metrics:


,task,examples,token_f1,rouge_l_f1,numeric_recall,unsupported_numeric_rate
0,cause_refusal,34,0.380922,0.319513,1.000000,0.000000
1,extent,34,0.390302,0.368121,0.794118,0.138389
2,forecast_refusal,34,0.407116,0.340091,1.000000,0.032038
3,impact_refusal,34,0.508093,0.389754,1.000000,0.000000
4,integrated_summary,34,0.787488,0.775141,0.750000,0.190056
5,review_policy,34,0.292023,0.221726,0.088235,0.070261
6,sensor_attribution,34,0.271186,0.237288,0.000000,0.000000
7,spatial_geometry,34,0.392421,0.369324,0.588235,0.355545
8,temporary_sensor_attribution,34,0.498660,0.430589,0.808824,0.084314
9,uncertainty,34,0.555450,0.509855,0.783088,0.286262


P18-C16 COMPLETE ✅


In [23]:
# ============================================================
# CELL: P18-C17 — QUALITATIVE VALIDATION SAMPLES
# ============================================================
sample_tasks = [
    "extent",
    "uncertainty",
    "sensor_attribution",
    "temporary_sensor_attribution",
    "integrated_summary",
    "review_policy",
    "cause_refusal",
    "impact_refusal",
    "forecast_refusal",
]

for task in sample_tasks:
    row = generation_df[
        generation_df["task"]
        == task
    ].iloc[0]

    print("=" * 112)
    print("TASK :", task)
    print("SCENE:", row["scene_id"])
    print()
    print("REFERENCE:")
    print(row["reference"])
    print()
    print("PREDICTION:")
    print(row["prediction"])
    print()

print("P18-C17 COMPLETE ✅")


TASK : extent
SCENE: Somalia_12849

REFERENCE:
FloodShift predicts temporary flooding over 20.68% of the sensor-covered scene, corresponding to 5.38 km². Permanent water accounts for 0.07% of the sensor-covered scene. These values are model-derived predictions, not ground truth.

PREDICTION:
The predicted temporary flood and permanent water extent is 20.68%. The permanent water extent is 0.07%, and 0.07% is covered by the frozen frozen review-10 sensor. The mean predictive entropy is 0.186.

TASK : uncertainty
SCENE: Somalia_12849

REFERENCE:
The mean normalized predictive entropy is 0.186. 38.72% of sensor-covered pixels exceed the frozen review-10 threshold, and 72.95% exceed the frozen review-20 threshold. Mean optical reliability P(clear) is 0.498, with 49.63% of covered pixels below P(clear)=0.50.

PREDICTION:
The mean predictive entropy is 0.186. The review-10 and review20 counts as 72.95% of sensor-covered pixels, while 72.95% counts as counterfactual sensor-covered pixels. Mean

In [24]:
# ============================================================
# CELL: P18-C18 — EXPORT COMPACT MODEL + METRICS
# ============================================================
GENERATIONS_CSV = (
    METRIC_ROOT
    / "FloodShift_09_validation_generations.csv"
)

GEN_METRICS_CSV = (
    METRIC_ROOT
    / "FloodShift_09_generation_metrics.csv"
)

TASK_METRICS_CSV = (
    METRIC_ROOT
    / "FloodShift_09_task_metrics.csv"
)

ABLATION_CSV = (
    METRIC_ROOT
    / "FloodShift_09_visual_ablation.csv"
)

HISTORY_JSON = (
    METRIC_ROOT
    / "FloodShift_09_training_history.json"
)

PROTOCOL_JSON = (
    OUTPUT_ROOT
    / "FloodShift_09_protocol_record.json"
)

generation_df.to_csv(
    GENERATIONS_CSV,
    index=False,
)

generation_metrics_df.to_csv(
    GEN_METRICS_CSV,
    index=False,
)

task_metric_df.to_csv(
    TASK_METRICS_CSV,
    index=False,
)

ablation_df.to_csv(
    ABLATION_CSV,
    index=False,
)

with open(HISTORY_JSON, "w") as f:
    json.dump(
        history,
        f,
        indent=2,
    )

TOKENIZER_DIR = (
    CHECKPOINT_ROOT
    / "tokenizer"
)

tokenizer.save_pretrained(
    TOKENIZER_DIR
)

LORA_DIR = (
    CHECKPOINT_ROOT
    / "flan_t5_lora"
)

text_model.save_pretrained(
    LORA_DIR
)

BRIDGE_PATH = (
    CHECKPOINT_ROOT
    / "FloodShift_09_GaussianVisualBridge.pt"
)

torch.save(
    {
        "state_dict": {
            k: v.detach().cpu()
            for k, v in visual_bridge.state_dict().items()
        },
        "vision_model_name": VISION_MODEL_NAME,
        "text_model_name": TEXT_MODEL_NAME,
        "feature_dim": feature_dim,
        "text_dim": TEXT_DIM,
        "latent_dim": LATENT_DIM,
        "num_visual_tokens": NUM_VISUAL_TOKENS,
        "best_epoch": best_epoch,
        "best_validation_loss": best_val_loss,
        "loss_weights": {
            "beta_kl": BETA_KL,
            "lambda_var_align": LAMBDA_VAR_ALIGN,
            "lambda_aux": LAMBDA_AUX,
        },
    },
    BRIDGE_PATH,
)

protocol09 = {
    "experiment": "FloodShift-09-Gaussian-Latent-Variational-VLM",
    "source_08b_archive_sha256": ARCHIVE_08B_SHA,
    "source_segmenter": "FloodShift-04H-CAL",
    "vision_backbone": VISION_MODEL_NAME,
    "vision_backbone_trainable": False,
    "text_backbone": TEXT_MODEL_NAME,
    "text_backbone_adaptation": "LoRA_q_v",
    "latent_distribution": "diagonal_Gaussian",
    "latent_dim": LATENT_DIM,
    "visual_tokens": NUM_VISUAL_TOKENS,
    "reparameterization": "z=mu+sigma*epsilon",
    "beta_kl": BETA_KL,
    "lambda_variance_alignment": LAMBDA_VAR_ALIGN,
    "lambda_visual_auxiliary": LAMBDA_AUX,
    "epochs": EPOCHS,
    "best_epoch": best_epoch,
    "best_validation_loss": best_val_loss,
    "validation_generation_metrics": summary_metrics,
    "visual_ablation": {
        "full_text_loss": full_val["text_loss"],
        "zero_text_loss": zero_val["text_loss"],
        "shuffled_text_loss": shuffled_val["text_loss"],
        "full_vs_zero_gain": (
            zero_val["text_loss"]
            - full_val["text_loss"]
        ),
        "full_vs_shuffled_gain": (
            shuffled_val["text_loss"]
            - full_val["text_loss"]
        ),
    },
    "latent_uncertainty": {
        "validation_MAE": full_val["latent_uncertainty_MAE"],
        "entropy_correlation": full_val["latent_entropy_correlation"],
    },
    "visual_auxiliary_MAE": {
        "temporary_flood_fraction": full_val["temp_fraction_MAE"],
        "predictive_entropy": full_val["entropy_MAE"],
        "optical_attribution_share": full_val["optical_share_MAE"],
    },
    "geographic_test_status": "LOCKED_NOT_EVALUATED",
}

with open(PROTOCOL_JSON, "w") as f:
    json.dump(
        protocol09,
        f,
        indent=2,
    )

print("=" * 112)
print("P18-C18 — EXPORT")
print("=" * 112)
print("LoRA adapter    :", LORA_DIR)
print("Gaussian bridge :", BRIDGE_PATH)
print("Tokenizer       :", TOKENIZER_DIR)
print("Generations     :", GENERATIONS_CSV)
print("Ablation        :", ABLATION_CSV)
print("Protocol        :", PROTOCOL_JSON)
print("Geo test        : LOCKED_NOT_EVALUATED")
print("P18-C18 COMPLETE ✅")


P18-C18 — EXPORT
LoRA adapter    : /kaggle/working/FloodShift_09_Variational_VLM/checkpoint/flan_t5_lora
Gaussian bridge : /kaggle/working/FloodShift_09_Variational_VLM/checkpoint/FloodShift_09_GaussianVisualBridge.pt
Tokenizer       : /kaggle/working/FloodShift_09_Variational_VLM/checkpoint/tokenizer
Generations     : /kaggle/working/FloodShift_09_Variational_VLM/metrics/FloodShift_09_validation_generations.csv
Ablation        : /kaggle/working/FloodShift_09_Variational_VLM/metrics/FloodShift_09_visual_ablation.csv
Protocol        : /kaggle/working/FloodShift_09_Variational_VLM/FloodShift_09_protocol_record.json
Geo test        : LOCKED_NOT_EVALUATED
P18-C18 COMPLETE ✅


In [25]:
# ============================================================
# CELL: P18-C19 — COMPACT ARCHIVE + SHA256
# ============================================================
def file_sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()

MANIFEST_CSV = (
    OUTPUT_ROOT
    / "FloodShift_09_file_manifest.csv"
)

manifest_rows = []

for root in [
    CHECKPOINT_ROOT,
    METRIC_ROOT,
]:
    for p in sorted(root.rglob("*")):
        if p.is_file():
            manifest_rows.append({
                "relative_path": str(
                    p.relative_to(OUTPUT_ROOT)
                ),
                "bytes": int(
                    p.stat().st_size
                ),
                "sha256": file_sha256(p),
            })

manifest_rows.append({
    "relative_path": str(
        PROTOCOL_JSON.relative_to(OUTPUT_ROOT)
    ),
    "bytes": int(
        PROTOCOL_JSON.stat().st_size
    ),
    "sha256": file_sha256(
        PROTOCOL_JSON
    ),
})

manifest_df = pd.DataFrame(
    manifest_rows
)

manifest_df.to_csv(
    MANIFEST_CSV,
    index=False,
)

ARCHIVE_PATH = (
    OUTPUT_ROOT
    / "FloodShift_09_ARCHIVE.zip"
)

with zipfile.ZipFile(
    ARCHIVE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for root in [
        CHECKPOINT_ROOT,
        METRIC_ROOT,
    ]:
        for p in sorted(root.rglob("*")):
            if p.is_file():
                zf.write(
                    p,
                    arcname=p.relative_to(
                        OUTPUT_ROOT
                    ),
                )

    zf.write(
        PROTOCOL_JSON,
        arcname=PROTOCOL_JSON.relative_to(
            OUTPUT_ROOT
        ),
    )

    zf.write(
        MANIFEST_CSV,
        arcname=MANIFEST_CSV.relative_to(
            OUTPUT_ROOT
        ),
    )

ARCHIVE_SHA = file_sha256(
    ARCHIVE_PATH
)

print("=" * 112)
print("P18-C19 — FINAL ARCHIVE")
print("=" * 112)
print("Best epoch         :", best_epoch)
print("Best val loss      :", best_val_loss)
print("Numeric recall     :", summary_metrics["numeric_recall"])
print("Unsupported nums   :", summary_metrics["unsupported_numeric_rate"])
print("Refusal recall     :", summary_metrics["refusal_recall"])
print("Full-zero gain     :", zero_val["text_loss"] - full_val["text_loss"])
print("Full-shuffle gain  :", shuffled_val["text_loss"] - full_val["text_loss"])
print("Latent-entropy r   :", full_val["latent_entropy_correlation"])
print("Manifest files     :", len(manifest_df))
print("Geo test           : LOCKED_NOT_EVALUATED")
print("Archive            :", ARCHIVE_PATH)
print("Archive SHA256     :", ARCHIVE_SHA)
print()
print("Send P18-C12 through P18-C19 outputs for acceptance analysis.")
print("P18-C19 COMPLETE ✅")


P18-C19 — FINAL ARCHIVE
Best epoch         : 3
Best val loss      : 0.7407236281563254
Numeric recall     : 0.68125
Unsupported nums   : 0.11568644590703414
Refusal recall     : 0.9705882352941176
Full-zero gain     : 0.009537395308999441
Full-shuffle gain  : -5.827230565746966e-06
Latent-entropy r   : 0.03812696081395254
Manifest files     : 15
Geo test           : LOCKED_NOT_EVALUATED
Archive            : /kaggle/working/FloodShift_09_Variational_VLM/FloodShift_09_ARCHIVE.zip
Archive SHA256     : 126bcac0ac4ae899ae6a3eb9cb4a47dad0ee6ac92cf22454d6737574713909cc

Send P18-C12 through P18-C19 outputs for acceptance analysis.
P18-C19 COMPLETE ✅


## After the run

Send me the executed notebook or the outputs from:

- **P18-C12** — all 3 epoch summaries
- **P18-C14** — full / zero / shuffled vision ablation
- **P18-C16** — grounded generation metrics
- **P18-C17** — qualitative samples
- **P18-C19** — archive SHA256

Download:

```text
FloodShift_09_ARCHIVE.zip
```

The archive intentionally stores only compact trainable artifacts:

- FLAN-T5 LoRA adapter
- Gaussian visual bridge
- tokenizer/config
- training history
- validation generations
- grounded-generation metrics
- visual-ablation metrics
- protocol record
- per-file SHA manifest

The frozen CLIP and FLAN-T5 base weights are not duplicated.

We will accept the variational VLM only after checking:

1. validation loss,
2. full vs zero/shuffled visual conditioning,
3. latent-entropy correlation,
4. visual auxiliary MAE,
5. numeric grounding,
6. refusal behavior,
7. unsupported-number hallucination.
